## Exploración inicial de los datasets

Objetivo: comprender la estructura de las tablas HECHOS y VICTIMAS, identificar su relación y evaluar la calidad inicial de los datos.


### Importando librerías

In [5]:
import sqlite3
print(sqlite3.sqlite_version)

3.37.2


In [6]:
import pandas as pd
import numpy as np


### Carga de datos

In [7]:
# Cargar los datasets originales

#hechos = pd.read_excel("../data_raw/siniestros_viales_hechos.xlsx")
#victimas = pd.read_excel("../data_raw/siniestros_viales_victimas.xlsx")

# Cargar la hoja principal de cada dataset original
hechos = pd.read_excel(
    "../data_raw/siniestros_viales_hechos.xlsx",
    sheet_name="HECHOS"
)

victimas = pd.read_excel(
    "../data_raw/siniestros_viales_victimas.xlsx",
    sheet_name="VICTIMAS"
)

### Dimensiones de los datasets

In [8]:
# Comparar cantidad de registros y columnas

print("HECHOS:", hechos.shape)
print("VICTIMAS:", victimas.shape)

HECHOS: (65818, 21)
VICTIMAS: (75193, 9)


Se observa que la tabla VICTIMAS contiene más registros que HECHOS, lo que sugiere que un mismo siniestro podría estar asociado a múltiples víctimas.

### Estructura de las tablas

In [9]:
# Mostrar columnas disponibles en cada dataset

print("COLUMNAS HECHOS")
print(hechos.columns.tolist())

print("\nCOLUMNAS VICTIMAS")
print(victimas.columns.tolist())

COLUMNAS HECHOS
['id_siniestro', 'numero_total_de_victimas', 'numero_victimas_leve_siniestro', 'numero_victimas_grave_siniestro', 'numero_victimas_mortal_siniestro', 'fecha_siniestro', 'anio_siniestro', 'mes_siniestro', 'dia_siniestro', 'hora_siniestro', 'rango_horario', 'direccion_normalizada_siniestro', 'comuna_siniestro', 'tipo_de_via_siniestro', 'geocodificacion_plana', 'longitud_siniestro', 'latitud_siniestro', 'participantes_siniestro', 'modo_desplazamiento_victima', 'contraparte_siniestro', 'gravedad_siniestro']

COLUMNAS VICTIMAS
['id_siniestro', 'fecha_siniestro', 'anio_siniestro', 'modo_desplazamiento_victima', 'sexo_victima', 'edad_victima', 'GRAVEdad_victima', 'rol_victima', 'fecha_fallecimiento_victima']


Ambas tablas contienen el campo id_siniestro, que será analizado como posible clave de relación.

### Relación entre las tablas

In [10]:
# Cantidad de siniestros únicos en cada tabla
print("HECHOS - ids únicos:", hechos["id_siniestro"].nunique())
print("VICTIMAS - ids únicos:", victimas["id_siniestro"].nunique())

HECHOS - ids únicos: 65818
VICTIMAS - ids únicos: 65818


In [11]:
# Verificar si existen siniestros repetidos en HECHOS
hechos["id_siniestro"].duplicated().sum()

np.int64(0)

In [12]:
# Contar siniestros repetidos en VICTIMAS
victimas["id_siniestro"].duplicated().sum()

np.int64(9375)

In [13]:
# Cantidad de víctimas por siniestro
victimas_por_siniestro = victimas["id_siniestro"].value_counts()

victimas_por_siniestro.head()

id_siniestro
LC-2024-0243962    26
LC-2024-0666832    19
LC-2022-0166804    18
LC-2019-0291069    16
LC-2019-0188476    16
Name: count, dtype: int64

Los resultados obtenidos muestran que:

- Ambas tablas contienen exactamente 65.818 siniestros únicos.
- No existen siniestros duplicados en la tabla HECHOS.
- La tabla VICTIMAS contiene 9.375 registros adicionales debido a la presencia de múltiples víctimas asociadas a un mismo siniestro.
- Se identificaron accidentes con hasta 26 víctimas registradas.

Por lo tanto, se confirma una relación de uno a muchos (1:N), donde un siniestro puede estar asociado a una o varias víctimas.

Esta conclusión resulta fundamental para las siguientes etapas del proyecto, ya que permitirá definir correctamente la unidad de análisis y la estrategia de integración entre ambas tablas.

Comprobando si hay más datos o información en los datasets

In [14]:
# Listar hojas del archivo HECHOS
xls_hechos = pd.ExcelFile("../data_raw/siniestros_viales_hechos.xlsx")
print(xls_hechos.sheet_names)

# Listar hojas del archivo VICTIMAS
xls_victimas = pd.ExcelFile("../data_raw/siniestros_viales_victimas.xlsx")
print(xls_victimas.sheet_names)

['HECHOS', 'DICCIONARIO_HECHOS']
['VICTIMAS', 'DICCIONARIO_VICTIMAS']


Diccionarios de datos

In [15]:
# Cargar diccionarios de datos
dic_hechos = pd.read_excel(
    "../data_raw/siniestros_viales_hechos.xlsx",
    sheet_name="DICCIONARIO_HECHOS"
)

dic_victimas = pd.read_excel(
    "../data_raw/siniestros_viales_victimas.xlsx",
    sheet_name="DICCIONARIO_VICTIMAS"
)


display(dic_hechos.head(22))
display(dic_victimas.head(10))


,Variables y definiciones,Descripción,Unnamed: 2
0,id_siniestro,Identificador unico del siniestro,NaN
1,numero_total_de_victimas,Cantidad de víctimas totales,NaN
2,numero_victimas_leve_siniestro,Cantidad de víctimas leves,NaN
3,numero_victimas_grave_siniestro,Cantidad de víctimas graves,NaN
4,numero_victimas_mortal_siniestro,Cantidad de víctimas mortales,NaN
5,fecha_siniestro,Fecha en formato aaaa-mm-dd,NaN
6,anio_siniestro,Año,NaN
7,mes_siniestro,Mes,NaN
8,dia_siniestro,Día del mes,NaN
9,hora_siniestro,Hora del siniestro,NaN


,Definiciones,Descripción,Unnamed: 2
0,id_siniestro,Identificador unico del siniestro,NaN
1,fecha_siniestro,Fecha en formato aaaa-mm-dd en la que sucedió ...,NaN
2,anio_siniestro,Año del siniestro,NaN
3,modo_desplazamiento_victima,Vehículo que ocupaba quien haya fallecido o se...,NaN
4,sexo_victima,Sexo de la víctima informado por fuente policial,NaN
5,edad_victima,Edad de la víctima al momento del siniestro,NaN
6,gravedad_victima,Nivel máximo conocido de gravedad de la lesión...,NaN
7,rol_victima,Posición relativa al vehículo que presentaba l...,NaN
8,fecha_fallecimiento_victima,Fecha de fallecimiento de las víctimas mortales,NaN
9,NaN,NaN,NaN


Los diccionarios permiten validar el significado de las variables y facilitarán la selección de atributos para el modelo en etapas posteriores.

### Calidad inicial de los datos

In [16]:
# Cantidad de valores nulos por columna
hechos.isnull().sum()

id_siniestro                            0
numero_total_de_victimas                0
numero_victimas_leve_siniestro          0
numero_victimas_grave_siniestro         0
numero_victimas_mortal_siniestro        0
fecha_siniestro                         0
anio_siniestro                          0
mes_siniestro                           0
dia_siniestro                           0
hora_siniestro                         77
rango_horario                          77
direccion_normalizada_siniestro     12900
comuna_siniestro                     3017
tipo_de_via_siniestro               12227
geocodificacion_plana                2367
longitud_siniestro                   2378
latitud_siniestro                    2378
participantes_siniestro                 0
modo_desplazamiento_victima             0
contraparte_siniestro                   0
gravedad_siniestro                      0
dtype: int64

In [17]:
# Cantidad de valores nulos por columna
victimas.isnull().sum()

id_siniestro                       0
fecha_siniestro                    0
anio_siniestro                     0
modo_desplazamiento_victima        0
sexo_victima                       0
edad_victima                       0
GRAVEdad_victima                   0
rol_victima                        0
fecha_fallecimiento_victima    74490
dtype: int64

Se identifican valores faltantes principalmente en algunas variables de la tabla HECHOS, especialmente en dirección, tipo de vía y datos de geolocalización.

En la tabla VICTIMAS, la única variable con una cantidad significativa de valores nulos es `fecha_fallecimiento_victima`. Dado que esta columna registra la fecha de fallecimiento de víctimas mortales, la ausencia de valor podría indicar que la víctima no falleció. Dicha  variable `fecha_fallecimiento_victima` surge como una posible fuente para identificar víctimas fatales. 

En una etapa posterior se evaluará el tratamiento más adecuado para cada uno de estos casos.

### Tipos de datos

In [18]:
# Revisar tipos de datos de HECHOS
hechos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 65818 entries, 0 to 65817
Data columns (total 21 columns):
 #   Column                            Non-Null Count  Dtype 
---  ------                            --------------  ----- 
 0   id_siniestro                      65818 non-null  object
 1   numero_total_de_victimas          65818 non-null  object
 2   numero_victimas_leve_siniestro    65818 non-null  int64 
 3   numero_victimas_grave_siniestro   65818 non-null  int64 
 4   numero_victimas_mortal_siniestro  65818 non-null  int64 
 5   fecha_siniestro                   65818 non-null  object
 6   anio_siniestro                    65818 non-null  int64 
 7   mes_siniestro                     65818 non-null  int64 
 8   dia_siniestro                     65818 non-null  int64 
 9   hora_siniestro                    65741 non-null  object
 10  rango_horario                     65741 non-null  object
 11  direccion_normalizada_siniestro   52918 non-null  object
 12  comuna_siniestro  

In [19]:
# Revisar tipos de datos de VICTIMAS
victimas.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 75193 entries, 0 to 75192
Data columns (total 9 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   id_siniestro                 75193 non-null  object
 1   fecha_siniestro              75193 non-null  object
 2   anio_siniestro               75193 non-null  int64 
 3   modo_desplazamiento_victima  75193 non-null  object
 4   sexo_victima                 75193 non-null  object
 5   edad_victima                 75193 non-null  object
 6   GRAVEdad_victima             75193 non-null  object
 7   rol_victima                  75193 non-null  object
 8   fecha_fallecimiento_victima  703 non-null    object
dtypes: int64(1), object(8)
memory usage: 5.2+ MB


Se observa que varias variables se encuentran almacenadas como tipo `object`, incluyendo fechas, coordenadas geográficas y edades.

Estas variables requerirán validación y transformación durante la etapa de preparación de datos para garantizar su correcta utilización en el análisis y el modelado.

Asimismo, la variable `fecha_fallecimiento_victima` presenta únicamente 703 registros con información disponible, por lo que surge como una posible fuente para identificar víctimas fatales.

In [20]:
# Cantidad de registros por gravedad de la víctima
victimas["GRAVEdad_victima"].value_counts(dropna=False)

GRAVEdad_victima
LEVE      71308
GRAVE      3182
MORTAL      612
mortal       91
Name: count, dtype: int64

La variable `GRAVEdad_victima` presenta cuatro categorías:

- LEVE
- GRAVE
- MORTAL
- mortal

Se observa una inconsistencia de formato entre las categorías `MORTAL` y `mortal`, que representan el mismo concepto y deberán ser unificadas durante la etapa de preparación de datos.

Asimismo, la suma de ambas categorías coincide con la cantidad de registros no nulos en `fecha_fallecimiento_victima` (703 casos), lo que refuerza la consistencia interna de los datos y permite identificar de forma confiable las víctimas fatales.